# Sample Audio / Video / Transcript Alignment Checks

This notebook samples short chunks from the `vad_new/{patient}/vad_data` interval folders so we can quickly compare:

- LR-ASD active-speaker video: `video_out.avi`
- existing transcription audio: `transcription/audio.wav`
- audio freshly extracted from stitched `NSP-1.ns5`, channel `2` (behavioral series, fewest channels)
- audio freshly extracted from stitched `NSP-2.ns5`, channel `2` (behavioral series, fewest channels)
- WhisperX transcript text overlapping the sampled window

The sampler is seedable. Set `RANDOM_SEED = None` for a different draw each run, or set an integer to reproduce the same samples.

In [50]:
from pathlib import Path
import json
import math
import random
import shutil
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
import quantities as pq
from IPython.display import Audio, HTML, Video, display
from neo.io import BlackrockIO


In [51]:
# --------------------
# User-facing settings
# --------------------
PATIENT = "YFU"
VAD_ROOT = Path(f"/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/{PATIENT}")
MANIFEST_PATH = VAD_ROOT / f"{PATIENT}_vad_merged_intervals.csv"
VAD_DATA_DIR = VAD_ROOT / "vad_data"

# Keep samples small and easy to inspect.
SAMPLE_SECONDS = 5.0

# One sample per recording / TOC by default. Increase for denser checks.
SAMPLES_PER_RECORDING = 1
MAX_RECORDINGS = None      # e.g. 10 for a quick pass, None for every recording with complete assets
MAX_SAMPLES_TOTAL = 30     # safety cap; set None to sample every eligible recording

# Keep this False for speed. Turn on only if older video folders use a different layout.
ALLOW_SLOW_RECURSIVE_VIDEO_FALLBACK = False

# Reproducibility knob. Use None to get a fresh random draw each run.
RANDOM_SEED = None

# Audio channel index within the behavioral (fewest-channel) analog signal.
# Same index for both NSP-1 and NSP-2 — both carry RoomMic1 at position 2.
NSP1_AUDIO_CHAN = 2
NSP2_AUDIO_CHAN = 2
ORIGINAL_FS = 30_000
TARGET_FS = 16_000

# Output goes beside this notebook, not into the large vad_data tree.
SAMPLE_ROOT = Path("/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples")
RUN_LABEL = f"{PATIENT}_seed-{RANDOM_SEED if RANDOM_SEED is not None else 'random'}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_DIR = SAMPLE_ROOT / RUN_LABEL

print("manifest:", MANIFEST_PATH)
print("sample output:", OUT_DIR)

manifest: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFU/YFU_vad_merged_intervals.csv
sample output: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440


In [52]:
def interval_dir(interval_id: str) -> Path:
    return VAD_DATA_DIR / interval_id


def first_existing(paths):
    for path in paths:
        if path and Path(path).exists():
            return Path(path)
    return None


def find_video_out(interval_path: Path) -> Path | None:
    # LR-ASD writes video_out.avi under:
    # video/{interval_id}/neural/{cam_serial}/synced_video/neural_{cam_serial}/pyavi/video_out.avi
    interval_id = interval_path.name
    direct_hits = sorted((interval_path / "video" / interval_id / "neural").glob("*/synced_video/*/pyavi/video_out.avi"))
    if direct_hits:
        return direct_hits[0]
    if ALLOW_SLOW_RECURSIVE_VIDEO_FALLBACK:
        hits = sorted(interval_path.glob("video/**/video_out.avi"))
        return hits[0] if hits else None
    return None


def find_synced_video(interval_path: Path) -> Path | None:
    interval_id = interval_path.name
    direct_hits = sorted((interval_path / "video" / interval_id / "neural").glob("*/synced_video/*.mp4"))
    if direct_hits:
        return direct_hits[0]
    if not ALLOW_SLOW_RECURSIVE_VIDEO_FALLBACK:
        return None
    hits = sorted(interval_path.glob("video/**/*.mp4"))
    if not hits:
        return None
    preferred = [p for p in hits if "synced_video" in str(p)]
    return preferred[0] if preferred else hits[0]


def find_neural_ns5(interval_path: Path, nsp: int) -> Path | None:
    hits = sorted((interval_path / "neural").glob(f"*_NSP-{nsp}.ns5"))
    return hits[0] if hits else None


def transcript_json_path(interval_path: Path) -> Path | None:
    path = interval_path / "transcription" / "whisperx" / "audio.json"
    return path if path.exists() else None


def transcription_audio_path(interval_path: Path) -> Path | None:
    path = interval_path / "transcription" / "audio.wav"
    return path if path.exists() else None


def load_whisper_segments(json_path: Path) -> list[dict]:
    if json_path is None or not json_path.exists():
        return []
    with open(json_path) as f:
        data = json.load(f)
    return data.get("segments", [])


def segment_text_window(segments: list[dict], start_s: float, end_s: float) -> tuple[str, list[dict]]:
    overlaps = []
    lines = []
    for seg in segments:
        seg_start = float(seg.get("start", 0.0))
        seg_end = float(seg.get("end", seg_start))
        if seg_end < start_s or seg_start > end_s:
            continue
        overlaps.append(seg)
        speaker = seg.get("speaker", "")
        text = str(seg.get("text", "")).strip()
        prefix = f"[{seg_start:8.3f}-{seg_end:8.3f}]"
        if speaker:
            prefix += f" {speaker}:"
        lines.append(f"{prefix} {text}")
    return "\n".join(lines), overlaps


In [53]:
def normalize_audio(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x).squeeze().astype(np.float32)
    if x.ndim != 1:
        raise ValueError(f"Expected 1D audio after channel selection, got shape {x.shape}")
    max_abs = np.max(np.abs(x)) if x.size else 0
    if max_abs > 0:
        x = x / max_abs
    return x


def resample_audio(x: np.ndarray, source_fs: int, target_fs: int) -> np.ndarray:
    x = normalize_audio(x)
    if source_fs == target_fs:
        return x
    tensor = torch.from_numpy(x)
    y = torchaudio.functional.resample(tensor, source_fs, target_fs).numpy()
    return normalize_audio(y)


def write_wav_window(source_wav: Path, out_wav: Path, start_s: float, duration_s: float, target_fs: int = TARGET_FS) -> dict:
    info = sf.info(str(source_wav))
    source_fs = int(info.samplerate)
    start_frame = max(0, int(round(start_s * source_fs)))
    n_frames = max(1, int(round(duration_s * source_fs)))
    audio, fs = sf.read(str(source_wav), start=start_frame, frames=n_frames, always_2d=False)
    if np.asarray(audio).ndim > 1:
        audio = np.asarray(audio)[:, 0]
    audio = resample_audio(audio, fs, target_fs)
    out_wav.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(out_wav), audio, target_fs, subtype="PCM_16")
    return {"source_fs": fs, "target_fs": target_fs, "n_samples": int(audio.shape[0]), "duration_s": float(audio.shape[0] / target_fs)}


def _pick_behavioral_signal(analogsignals):
    """Return the analog signal with the fewest channels (behavioral/audio series).

    Both NSP-1 and NSP-2 contain a neural signal (many channels) and a behavioral
    signal (fewer channels, carries audio/sync).  Picking fewest channels works for
    both NSP layouts without relying on signal name strings.
    """
    return min(analogsignals, key=lambda s: s.shape[1] if len(s.shape) > 1 else 1)


def _load_signal_window(signal_proxy, audio_chan: int, rel_start_s: float, rel_end_s: float, original_fs: int):
    # Try Neo's time_slice first, then fall back to loading the segment and slicing by sample index.
    signal_t0_s = float(signal_proxy.t_start)
    try:
        sliced = signal_proxy.load(time_slice=((signal_t0_s + rel_start_s) * pq.s, (signal_t0_s + rel_end_s) * pq.s))
        arr = np.asarray(sliced[:, audio_chan]).squeeze()
    except Exception:
        loaded = signal_proxy.load()
        start_idx = max(0, int(round(rel_start_s * original_fs)))
        end_idx = max(start_idx + 1, int(round(rel_end_s * original_fs)))
        arr = np.asarray(loaded[start_idx:end_idx, audio_chan]).squeeze()
    return arr


def extract_nsx_audio_window(ns5_path: Path, audio_chan: int, out_wav: Path, start_s: float, duration_s: float, original_fs: int = ORIGINAL_FS, target_fs: int = TARGET_FS) -> dict:
    reader = BlackrockIO(str(ns5_path))
    n_blocks = reader.header["nb_block"]
    n_segments = reader.header["nb_segment"]

    target_start = float(start_s)
    target_end = float(start_s + duration_s)
    first_signal_t0 = None
    cursor = target_start
    pieces = []

    for block_idx in range(n_blocks):
        block = reader.read_block(block_index=block_idx, lazy=True)
        for segment_idx in range(n_segments[block_idx]):
            segment = block.segments[segment_idx]
            if not segment.analogsignals:
                continue
            analog_signal = _pick_behavioral_signal(segment.analogsignals)

            signal_t0 = float(analog_signal.t_start)
            if first_signal_t0 is None:
                first_signal_t0 = signal_t0

            rel_segment_start = signal_t0 - first_signal_t0
            rel_segment_end = rel_segment_start + (analog_signal.shape[0] / original_fs)

            overlap_start = max(target_start, rel_segment_start)
            overlap_end = min(target_end, rel_segment_end)
            if overlap_end <= overlap_start:
                continue

            if overlap_start > cursor:
                gap_n = int(round((overlap_start - cursor) * original_fs))
                if gap_n > 0:
                    pieces.append(np.zeros(gap_n, dtype=np.float32))

            local_start = overlap_start - rel_segment_start
            local_end = overlap_end - rel_segment_start
            pieces.append(_load_signal_window(analog_signal, audio_chan, local_start, local_end, original_fs))
            cursor = overlap_end

    if cursor < target_end:
        gap_n = int(round((target_end - cursor) * original_fs))
        if gap_n > 0:
            pieces.append(np.zeros(gap_n, dtype=np.float32))

    if not pieces:
        raise RuntimeError(f"No overlapping analog data found in {ns5_path} for {start_s:.3f}-{start_s + duration_s:.3f}s")

    audio = np.concatenate([np.asarray(p).squeeze() for p in pieces]).astype(np.float32)
    audio = resample_audio(audio, original_fs, target_fs)
    out_wav.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(out_wav), audio, target_fs, subtype="PCM_16")
    return {"source_fs": original_fs, "target_fs": target_fs, "n_samples": int(audio.shape[0]), "duration_s": float(audio.shape[0] / target_fs)}

In [54]:
def run_command(cmd: list[str]) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, capture_output=True, text=True)


def make_video_clip(source_video: Path, out_video: Path, start_s: float, duration_s: float) -> dict:
    out_video.parent.mkdir(parents=True, exist_ok=True)

    # First try stream copy for speed. If the container/codecs do not allow it, re-encode.
    fast_cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-ss", f"{start_s:.3f}", "-i", str(source_video),
        "-t", f"{duration_s:.3f}", "-c", "copy", str(out_video),
    ]
    res = run_command(fast_cmd)
    if res.returncode == 0 and out_video.exists() and out_video.stat().st_size > 0:
        return {"method": "stream_copy", "returncode": res.returncode}

    reencode_cmd = [
        "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
        "-ss", f"{start_s:.3f}", "-i", str(source_video),
        "-t", f"{duration_s:.3f}",
        "-c:v", "mpeg4", "-q:v", "4", "-c:a", "pcm_s16le", str(out_video),
    ]
    res2 = run_command(reencode_cmd)
    if res2.returncode != 0:
        raise RuntimeError(f"ffmpeg failed for {source_video}\nfast stderr: {res.stderr}\nreencode stderr: {res2.stderr}")
    return {"method": "reencode", "returncode": res2.returncode}


def choose_sample_start(row: pd.Series, segments: list[dict], rng: random.Random) -> float:
    duration = float(row["duration_s"])
    latest_start = max(0.0, duration - SAMPLE_SECONDS)
    usable_segments = []
    for seg in segments:
        seg_start = float(seg.get("start", 0.0))
        seg_end = float(seg.get("end", seg_start))
        if seg_end <= 0 or seg_start >= duration:
            continue
        if seg_end - seg_start >= 0.5:
            usable_segments.append((max(0.0, seg_start), min(duration, seg_end)))

    if usable_segments:
        seg_start, seg_end = rng.choice(usable_segments)
        center = (seg_start + seg_end) / 2.0
        jitter = rng.uniform(-1.0, 1.0)
        return min(latest_start, max(0.0, center - SAMPLE_SECONDS / 2.0 + jitter))

    return rng.uniform(0.0, latest_start) if latest_start > 0 else 0.0


def build_candidate_table(manifest: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in manifest.iterrows():
        iid = str(row["interval_id"])
        idir = interval_dir(iid)
        t_audio = transcription_audio_path(idir)
        whisper_json = transcript_json_path(idir)
        video_out = find_video_out(idir)
        synced_video = find_synced_video(idir)
        nsp1 = find_neural_ns5(idir, 1)
        nsp2 = find_neural_ns5(idir, 2)
        rows.append({
            "interval_id": iid,
            "toc": row["toc"],
            "duration_s": float(row["duration_s"]),
            "interval_dir": idir,
            "video_out": video_out,
            "synced_video": synced_video,
            "transcription_audio": t_audio,
            "whisper_json": whisper_json,
            "nsp1_ns5": nsp1,
            "nsp2_ns5": nsp2,
            "stitch_success": (idir / "_SUCCESS").exists(),
            "transcription_success": (idir / "transcription" / "_SUCCESS").exists(),
        })
    candidates = pd.DataFrame(rows)
    candidates["has_all_assets"] = (
        candidates["stitch_success"]
        & candidates["transcription_success"]
        & candidates["video_out"].notna()
        & candidates["transcription_audio"].notna()
        & candidates["whisper_json"].notna()
        & candidates["nsp1_ns5"].notna()
        & candidates["nsp2_ns5"].notna()
        & (candidates["duration_s"] >= SAMPLE_SECONDS)
    )
    return candidates


In [55]:
manifest = pd.read_csv(MANIFEST_PATH)
print(f"manifest rows: {len(manifest)}")

candidates = build_candidate_table(manifest)
print("asset counts:")
for col in ["stitch_success", "transcription_success", "video_out", "transcription_audio", "whisper_json", "nsp1_ns5", "nsp2_ns5", "has_all_assets"]:
    if candidates[col].dtype == bool:
        print(f"  {col}: {int(candidates[col].sum())}")
    else:
        print(f"  {col}: {int(candidates[col].notna().sum())}")

eligible = candidates[candidates["has_all_assets"]].copy()
print(f"eligible intervals: {len(eligible)} across {eligible['toc'].nunique()} recordings/TOCs")
eligible.head()


manifest rows: 1299
asset counts:
  stitch_success: 1186
  transcription_success: 1186
  video_out: 1016
  transcription_audio: 1186
  whisper_json: 1186
  nsp1_ns5: 1186
  nsp2_ns5: 1186
  has_all_assets: 763
eligible intervals: 763 across 14 recordings/TOCs


,interval_id,toc,duration_s,interval_dir,video_out,synced_video,transcription_audio,whisper_json,nsp1_ns5,nsp2_ns5,stitch_success,transcription_success,has_all_assets
57,20251208-205741_0057,20251208-205741,33.1,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True
62,20251209-114622_0004,20251209-114622,10.6,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True
65,20251209-114622_0007,20251209-114622,34.1,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True
66,20251209-114622_0008,20251209-114622,103.0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True
68,20251209-114622_0010,20251209-114622,131.9,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,True,True,True


In [56]:
rng = random.Random(RANDOM_SEED)

tocs = list(eligible["toc"].drop_duplicates())
rng.shuffle(tocs)
if MAX_RECORDINGS is not None:
    tocs = tocs[:MAX_RECORDINGS]

selected_rows = []
for toc in tocs:
    group = eligible[eligible["toc"] == toc].copy()
    group_indices = list(group.index)
    rng.shuffle(group_indices)
    for idx in group_indices[:SAMPLES_PER_RECORDING]:
        selected_rows.append(eligible.loc[idx])

if MAX_SAMPLES_TOTAL is not None and len(selected_rows) > MAX_SAMPLES_TOTAL:
    rng.shuffle(selected_rows)
    selected_rows = selected_rows[:MAX_SAMPLES_TOTAL]

selected = pd.DataFrame(selected_rows).reset_index(drop=True)
print(f"selected samples: {len(selected)}")
selected[["interval_id", "toc", "duration_s", "video_out"]].head(20)


selected samples: 14


,interval_id,toc,duration_s,video_out
0,20251209-114622_0018,20251209-114622,54.900,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,20251217-095026_0168,20251217-095026,11.300,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,20251212-093455_0021,20251212-093455,40.000,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,20251218-092859_0050,20251218-092859,27.800,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,20251211-131720_0026,20251211-131720,20.200,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
5,20251214-100028_0001,20251214-100028,18.400,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
6,20251218-171657_0000,20251218-171657,671.115,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
7,20251208-205741_0057,20251208-205741,33.100,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
8,20251215-095457_0001,20251215-095457,62.400,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
9,20251213-140204_0064,20251213-140204,47.300,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


In [57]:
def write_sample_bundle(row: pd.Series, sample_idx: int, rng: random.Random) -> dict:
    interval_id = row["interval_id"]
    sample_dir = OUT_DIR / f"sample_{sample_idx:03d}_{interval_id}"
    sample_dir.mkdir(parents=True, exist_ok=True)

    segments = load_whisper_segments(Path(row["whisper_json"]))
    start_s = choose_sample_start(row, segments, rng)
    end_s = start_s + SAMPLE_SECONDS

    results = {
        "sample_idx": sample_idx,
        "patient": PATIENT,
        "interval_id": interval_id,
        "toc": row["toc"],
        "interval_duration_s": float(row["duration_s"]),
        "sample_start_s": float(start_s),
        "sample_end_s": float(end_s),
        "sample_duration_s": float(SAMPLE_SECONDS),
        "source_interval_dir": str(row["interval_dir"]),
        "source_video_out": str(row["video_out"]),
        "source_synced_video": str(row["synced_video"]),
        "source_transcription_audio": str(row["transcription_audio"]),
        "source_whisper_json": str(row["whisper_json"]),
        "source_nsp1_ns5": str(row["nsp1_ns5"]),
        "source_nsp2_ns5": str(row["nsp2_ns5"]),
    }

    results["video_out_sample"] = str(sample_dir / "video_out_sample.avi")
    results["video_clip_info"] = make_video_clip(Path(row["video_out"]), sample_dir / "video_out_sample.avi", start_s, SAMPLE_SECONDS)

    results["transcription_audio_sample"] = str(sample_dir / "transcription_audio_sample.wav")
    results["transcription_audio_info"] = write_wav_window(Path(row["transcription_audio"]), sample_dir / "transcription_audio_sample.wav", start_s, SAMPLE_SECONDS)

    results["nsp1_ch2_sample"] = str(sample_dir / "nsp1_ch2_sample.wav")
    results["nsp1_audio_info"] = extract_nsx_audio_window(Path(row["nsp1_ns5"]), NSP1_AUDIO_CHAN, sample_dir / "nsp1_ch2_sample.wav", start_s, SAMPLE_SECONDS)

    results["nsp2_ch2_sample"] = str(sample_dir / "nsp2_ch2_sample.wav")
    results["nsp2_audio_info"] = extract_nsx_audio_window(Path(row["nsp2_ns5"]), NSP2_AUDIO_CHAN, sample_dir / "nsp2_ch2_sample.wav", start_s, SAMPLE_SECONDS)

    transcript_text, transcript_segments = segment_text_window(segments, start_s, end_s)
    (sample_dir / "whisperx_window.txt").write_text(transcript_text if transcript_text else "[no WhisperX segment overlapped this window]\n")
    with open(sample_dir / "whisperx_window.json", "w") as f:
        json.dump(transcript_segments, f, indent=2)
    results["whisperx_window_txt"] = str(sample_dir / "whisperx_window.txt")
    results["whisperx_window_json"] = str(sample_dir / "whisperx_window.json")

    readme = f"""Alignment sample {sample_idx:03d}

Patient: {PATIENT}
Interval: {interval_id}
TOC / recording: {row['toc']}
Window: {start_s:.3f}s to {end_s:.3f}s within interval

Files to compare:
- video_out_sample.avi: LR-ASD active-speaker video crop
- transcription_audio_sample.wav: existing transcription/audio.wav crop
- nsp1_ch2_sample.wav: freshly extracted from stitched NSP-1.ns5, behavioral signal ch {NSP1_AUDIO_CHAN}
- nsp2_ch2_sample.wav: freshly extracted from stitched NSP-2.ns5, behavioral signal ch {NSP2_AUDIO_CHAN}
- whisperx_window.txt: WhisperX text overlapping this window

Source interval folder:
{row['interval_dir']}
"""
    (sample_dir / "README.txt").write_text(readme)

    with open(sample_dir / "sample_metadata.json", "w") as f:
        json.dump(results, f, indent=2, default=str)

    return results

In [58]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("writing samples to", OUT_DIR)

records = []
errors = []
for sample_idx, row in selected.iterrows():
    try:
        record = write_sample_bundle(row, sample_idx, rng)
        records.append(record)
        print(f"ok sample {sample_idx:03d}: {row['interval_id']} @ {record['sample_start_s']:.3f}s")
    except Exception as exc:
        errors.append({"sample_idx": int(sample_idx), "interval_id": row.get("interval_id"), "error": repr(exc)})
        print(f"FAILED sample {sample_idx:03d}: {row.get('interval_id')} -> {exc}")

summary = pd.DataFrame(records)
error_df = pd.DataFrame(errors)
summary_path = OUT_DIR / "sample_summary.csv"
summary.to_csv(summary_path, index=False)
if not error_df.empty:
    error_df.to_csv(OUT_DIR / "sample_errors.csv", index=False)

print(f"wrote {len(summary)} samples")
print("summary:", summary_path)
if errors:
    print("errors:", OUT_DIR / "sample_errors.csv")
summary.head()


writing samples to /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440
ok sample 000: 20251209-114622_0018 @ 32.325s
ok sample 001: 20251217-095026_0168 @ 1.651s
ok sample 002: 20251212-093455_0021 @ 0.000s
ok sample 003: 20251218-092859_0050 @ 9.710s
ok sample 004: 20251211-131720_0026 @ 0.000s
ok sample 005: 20251214-100028_0001 @ 3.297s
ok sample 006: 20251218-171657_0000 @ 445.753s
ok sample 007: 20251208-205741_0057 @ 2.100s
ok sample 008: 20251215-095457_0001 @ 52.258s
ok sample 009: 20251213-140204_0064 @ 33.493s
ok sample 010: 20251216-103510_0071 @ 426.410s
ok sample 011: 20251211-100159_0001 @ 150.114s
ok sample 012: 20251212-141804_0032 @ 239.110s
ok sample 013: 20251214-105634_0033 @ 493.352s
wrote 14 samples
summary: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_summary.csv


,sample_idx,patient,interval_id,toc,interval_duration_s,sample_start_s,sample_end_s,sample_duration_s,source_interval_dir,source_video_out,...,video_out_sample,video_clip_info,transcription_audio_sample,transcription_audio_info,nsp1_ch2_sample,nsp1_audio_info,nsp2_ch2_sample,nsp2_audio_info,whisperx_window_txt,whisperx_window_json
0,0,YFU,20251209-114622_0018,20251209-114622,54.9,32.325406,37.325406,5.0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,...,/scratch/tahaismail424/speech_247/notebooks/RE...,"{'method': 'stream_copy', 'returncode': 0}",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 16000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,/scratch/tahaismail424/speech_247/notebooks/RE...
1,1,YFU,20251217-095026_0168,20251217-095026,11.3,1.650706,6.650706,5.0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,...,/scratch/tahaismail424/speech_247/notebooks/RE...,"{'method': 'stream_copy', 'returncode': 0}",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 16000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,/scratch/tahaismail424/speech_247/notebooks/RE...
2,2,YFU,20251212-093455_0021,20251212-093455,40.0,0.000000,5.000000,5.0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,...,/scratch/tahaismail424/speech_247/notebooks/RE...,"{'method': 'stream_copy', 'returncode': 0}",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 16000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,/scratch/tahaismail424/speech_247/notebooks/RE...
3,3,YFU,20251218-092859_0050,20251218-092859,27.8,9.710113,14.710113,5.0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,...,/scratch/tahaismail424/speech_247/notebooks/RE...,"{'method': 'stream_copy', 'returncode': 0}",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 16000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,/scratch/tahaismail424/speech_247/notebooks/RE...
4,4,YFU,20251211-131720_0026,20251211-131720,20.2,0.000000,5.000000,5.0,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,...,/scratch/tahaismail424/speech_247/notebooks/RE...,"{'method': 'stream_copy', 'returncode': 0}",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 16000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,"{'source_fs': 30000, 'target_fs': 16000, 'n_sa...",/scratch/tahaismail424/speech_247/notebooks/RE...,/scratch/tahaismail424/speech_247/notebooks/RE...


In [59]:
# Quick inline review for the first few samples.
# Jupyter may not render AVI in all browsers, but the file links/paths are still printed.

N_DISPLAY = min(5, len(summary)) if "summary" in globals() else 0
for _, rec in summary.head(N_DISPLAY).iterrows():
    print("=" * 100)
    print(f"sample {rec['sample_idx']:03d} | {rec['interval_id']} | {rec['sample_start_s']:.3f}-{rec['sample_end_s']:.3f}s")
    print("folder:", Path(rec["video_out_sample"]).parent)
    print("video_out:", rec["video_out_sample"])
    print("transcription audio:", rec["transcription_audio_sample"])
    print("NSP-1 ch2:", rec["nsp1_ch2_sample"])
    print("NSP-2 ch2:", rec["nsp2_ch2_sample"])
    print("transcript:")
    print(Path(rec["whisperx_window_txt"]).read_text())
    display(Audio(filename=rec["transcription_audio_sample"]))
    display(Audio(filename=rec["nsp1_ch2_sample"]))
    display(Audio(filename=rec["nsp2_ch2_sample"]))
    try:
        display(Video(rec["video_out_sample"], embed=False, width=640))
    except Exception as exc:
        print("Video display failed, open the file path directly:", exc)

sample 000 | 20251209-114622_0018 | 32.325-37.325s
folder: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_000_20251209-114622_0018
video_out: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_000_20251209-114622_0018/video_out_sample.avi
transcription audio: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_000_20251209-114622_0018/transcription_audio_sample.wav
NSP-1 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_000_20251209-114622_0018/nsp1_ch2_sample.wav
NSP-2 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_000_20251209-114622_0018/nsp2_ch2_sample.wav
transcript:
[  32.72

sample 001 | 20251217-095026_0168 | 1.651-6.651s
folder: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_001_20251217-095026_0168
video_out: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_001_20251217-095026_0168/video_out_sample.avi
transcription audio: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_001_20251217-095026_0168/transcription_audio_sample.wav
NSP-1 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_001_20251217-095026_0168/nsp1_ch2_sample.wav
NSP-2 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_001_20251217-095026_0168/nsp2_ch2_sample.wav
transcript:
[   1.014-

sample 002 | 20251212-093455_0021 | 0.000-5.000s
folder: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_002_20251212-093455_0021
video_out: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_002_20251212-093455_0021/video_out_sample.avi
transcription audio: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_002_20251212-093455_0021/transcription_audio_sample.wav
NSP-1 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_002_20251212-093455_0021/nsp1_ch2_sample.wav
NSP-2 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_002_20251212-093455_0021/nsp2_ch2_sample.wav
transcript:
[   0.051-

sample 003 | 20251218-092859_0050 | 9.710-14.710s
folder: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_003_20251218-092859_0050
video_out: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_003_20251218-092859_0050/video_out_sample.avi
transcription audio: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_003_20251218-092859_0050/transcription_audio_sample.wav
NSP-1 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_003_20251218-092859_0050/nsp1_ch2_sample.wav
NSP-2 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_003_20251218-092859_0050/nsp2_ch2_sample.wav
transcript:
[   7.740

sample 004 | 20251211-131720_0026 | 0.000-5.000s
folder: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_004_20251211-131720_0026
video_out: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_004_20251211-131720_0026/video_out_sample.avi
transcription audio: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_004_20251211-131720_0026/transcription_audio_sample.wav
NSP-1 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_004_20251211-131720_0026/nsp1_ch2_sample.wav
NSP-2 ch2: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/speech_extraction/alignment_samples/YFU_seed-random_20260609_150440/sample_004_20251211-131720_0026/nsp2_ch2_sample.wav
transcript:
[   0.051-